### Personal AI Profile Chatbot
Tool: Pushover
- Pushover is a nifty tool for sending Push Notifications to your phone.
- Simply visit https://pushover.net/ and sign up for a free account, and create your API key.

Add to your .env file:
PUSHOVER_USER=
PUSHOVER_TOKEN=

Then install the app on your phone.

In [1]:
# Import necessary libraries
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from PyPDF2 import PdfReader
import gradio as gr

In [2]:
# Initialize OpenAI API client
load_dotenv(override=True)
openai = OpenAI()

In [3]:
# Pushover API configuration
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_api_url = "https://api.pushover.net/1/messages.json"

In [4]:
# Function to send push notifications using Pushover
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_api_url, data=payload)

In [5]:
# Test the push function
push("HEY!!")

Push: HEY!!


In [6]:
# Example function to record user details and send a push notification
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [7]:
# Example function to record unknown questions and send a push notification
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [8]:
# Define the JSON schema for the record_user_details function
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if this provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any other notes about this user that I should record"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about this conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [9]:
# Define the JSON schema for the record_unknown_question function
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any questions that I couldn't answered as you don't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [10]:
# List of tools with their corresponding JSON schemas
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if this provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about this conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any questions that I couldn't answered as you don't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['quest

In [12]:
# Function to handle tool calls based on the tool name and arguments
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT TO CALL THE RIGHT FUNCTION BASED ON THE TOOL NAME

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [13]:
# Example usage of the record_unknown_question function
globals()["record_unknown_question"]("this is a really hard question")

Push: Recording this is a really hard question asked that I couldn't answer


{'recorded': 'ok'}

In [14]:
# This is a more elegant way that avoids this IF statement
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [15]:
reader = PdfReader("/workspaces/ai-agents-lab/1_foundations/me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

# Read the summary from the text file
with open("/workspaces/ai-agents-lab/1_foundations/me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Ojo Gbenga Charles"

In [16]:
# Create the system prompt for the chatbot
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [17]:
# Function to handle the chat interactions with the user, utilizing the system prompt and tools
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        #This is the call to the LLM - see that we pass in the tools json
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason

        # If the LLM wants to call a tool, we do that!
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [ ]:
# Gradio interface to interact with the chatbot
gr.ChatInterface(chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d41dd4c24f33c5b536.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Tool called: record_unknown_question
Push: Recording do you have a patent? asked that I couldn't answer
Tool called: record_unknown_question
Push: Recording who's your favorite musician? asked that I couldn't answer
Tool called: record_user_details
Push: Recording interest from Name not provided with email gbe01nga@gmail.com and notes not provided


### Deployment